In [ ]:
!pip install osmnx -q
import ee
import osmnx as ox
import geopandas as gpd
import pandas as pd
import numpy as np
import warnings
from shapely.geometry import box

warnings.filterwarnings('ignore')

print("[LOG] Inițializare și Autentificare Google Earth Engine...")
ee.Authenticate()
ee.Initialize(project='mede-511511')
print("[LOG] GEE Inițializat cu succes.\n")

# ==========================================
# 0. GENERARE GRID (500m) - Unitatea de analiză
# ==========================================
print("[LOG] 0. Se generează gridul de 500m pentru București...")
bbox = [25.99, 44.38, 26.18, 44.50] # [Vest, Sud, Est, Nord]
roi = ee.Geometry.Rectangle(bbox)
projection = ee.Projection('EPSG:32635')

grid_500m = ee.Geometry.Rectangle(bbox).coveringGrid(projection, 500)
aria_celula_m2 = 500 * 500
print("[LOG] 0. Grid generat pe serverul GEE.\n")

# ==========================================
# 1. SURSA 1: LANDSAT 9 (Termic - LST)
# ==========================================
print("[LOG] 1. Interogare GEE pentru Landsat 9 LST...")
def extract_lst(image):
    thermal = image.select('ST_B10').multiply(0.00341802).add(149.0).subtract(273.15)
    return image.addBands(thermal.rename('LST_Celsius'))

l9_mean = (ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
           .filterBounds(roi)
           .filterDate('2026-06-01', '2026-08-31')
           .filter(ee.Filter.lt('CLOUD_COVER', 10))
           .map(extract_lst)
           .select('LST_Celsius')
           .mean())

grid_gee = l9_mean.reduceRegions(collection=grid_500m, reducer=ee.Reducer.mean(), scale=30)\
                  .map(lambda f: f.set('LST_Mediu_C', f.get('mean')))
print("[LOG] 1. Landsat 9 procesat.\n")

# ==========================================
# 2. SURSA 2: SENTINEL-2 (Optic - NDVI)
# ==========================================
print("[LOG] 2. Interogare GEE pentru Sentinel-2 NDVI...")
def add_ndvi(image):
    return image.addBands(image.normalizedDifference(['B8', 'B4']).rename('NDVI'))

s2_mean = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
           .filterBounds(roi)
           .filterDate('2026-06-01', '2026-08-31')
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
           .map(add_ndvi)
           .select('NDVI')
           .mean())

grid_gee = s2_mean.reduceRegions(collection=grid_gee, reducer=ee.Reducer.mean(), scale=10)\
                  .map(lambda f: f.set('NDVI_Mediu', f.get('mean')))
print("[LOG] 2. Sentinel-2 procesat.\n")

# ==========================================
# 3. SURSA 3a: GHSL (Înălțime Clădiri 2018)
# ==========================================
print("[LOG] 3. Interogare GEE pentru GHSL (Înălțime clădiri)...")
ghsl_height = ee.Image("JRC/GHSL/P2023A/GHS_BUILT_H/2018").select('built_height')
grid_gee = ghsl_height.reduceRegions(collection=grid_gee, reducer=ee.Reducer.mean(), scale=100)\
                      .map(lambda f: f.set('Inaltime_GHSL_m', f.get('mean')))
print("[LOG] 3. GHSL procesat.\n")

# ==========================================
# 4. EXTRAGERE GRID CĂTRE GEOPANDAS
# ==========================================
print("[LOG] 4. Se descarcă grid-ul satelitar din GEE în mediul local...")

grid_gee = grid_gee.map(lambda f: ee.Feature(f).transform('EPSG:4326', 0.001))
features = grid_gee.getInfo()['features']

grid_gdf = gpd.GeoDataFrame.from_features(features)
grid_gdf = grid_gdf.set_crs("EPSG:4326", allow_override=True).to_crs("EPSG:32635")
grid_gdf['ID_Celula'] = range(len(grid_gdf))
grid_gdf['geometry'] = grid_gdf.geometry.buffer(0)

for col in ['mean']:
    if col in grid_gdf.columns:
        grid_gdf = grid_gdf.drop(columns=[col])
print(f"[LOG] 4. Descărcare completă. Tabel de bază: {len(grid_gdf)} rânduri (Proiecție curentă: {grid_gdf.crs}).\n")

# ==========================================
# 5. SURSA 3b: OSMNX (Formă Urbană - Clădiri, Parcuri, Drumuri)
# ==========================================
print("[LOG] 5. Începe interogarea vectorială prin OSMnx...")
roi_poly = box(bbox[0], bbox[1], bbox[2], bbox[3])

print("[LOG] 5.1. Descărcare clădiri de pe Overpass API...")
osm_buildings = ox.features_from_polygon(polygon=roi_poly, tags={'building': True}).to_crs("EPSG:32635")
osm_buildings = osm_buildings[osm_buildings.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index()
osm_buildings['geometry'] = osm_buildings.geometry.buffer(0)

print("[LOG] 5.1. Intersecție spațială clădiri vs. grid...")
intersect_bld = gpd.overlay(grid_gdf[['ID_Celula', 'geometry']], osm_buildings[['geometry']], how='intersection')
intersect_bld['aria'] = intersect_bld.geometry.area
aria_bld = intersect_bld.groupby('ID_Celula')['aria'].sum().reset_index(name='Aria_Cladiri_m2')

print("[LOG] 5.2. Descărcare parcuri de pe Overpass API...")
osm_parks = ox.features_from_polygon(polygon=roi_poly, tags={'leisure': 'park', 'landuse': ['grass', 'forest']}).to_crs("EPSG:32635")
osm_parks = osm_parks[osm_parks.geom_type.isin(['Polygon', 'MultiPolygon'])].reset_index()
osm_parks['geometry'] = osm_parks.geometry.buffer(0)

print("[LOG] 5.2. Intersecție spațială parcuri vs. grid...")
intersect_prk = gpd.overlay(grid_gdf[['ID_Celula', 'geometry']], osm_parks[['geometry']], how='intersection')
intersect_prk['aria'] = intersect_prk.geometry.area
aria_prk = intersect_prk.groupby('ID_Celula')['aria'].sum().reset_index(name='Aria_Parcuri_m2')

print("[LOG] 5.3. Descărcare drumuri de pe Overpass API...")
osm_roads = ox.features_from_polygon(polygon=roi_poly, tags={'highway': True}).to_crs("EPSG:32635")
osm_roads = osm_roads[osm_roads.geom_type.isin(['LineString', 'MultiLineString'])].reset_index()

print("[LOG] 5.3. Intersecție spațială drumuri vs. grid...")
intersect_rds = gpd.overlay(grid_gdf[['ID_Celula', 'geometry']], osm_roads[['geometry']], how='intersection', keep_geom_type=False)
intersect_rds['lungime'] = intersect_rds.geometry.length
lungime_rds = intersect_rds.groupby('ID_Celula')['lungime'].sum().reset_index(name='Lungime_Drumuri_m')

print("[LOG] 5.4. Integrare trăsături OSM în tabelul aliniat...")
grid_gdf = grid_gdf.merge(aria_bld, on='ID_Celula', how='left').fillna({'Aria_Cladiri_m2': 0})
grid_gdf = grid_gdf.merge(aria_prk, on='ID_Celula', how='left').fillna({'Aria_Parcuri_m2': 0})
grid_gdf = grid_gdf.merge(lungime_rds, on='ID_Celula', how='left').fillna({'Lungime_Drumuri_m': 0})

grid_gdf['Fractie_Cladiri_OSM'] = grid_gdf['Aria_Cladiri_m2'] / aria_celula_m2
grid_gdf['Fractie_Parcuri_OSM'] = grid_gdf['Aria_Parcuri_m2'] / aria_celula_m2
grid_gdf['Densitate_Drumuri'] = grid_gdf['Lungime_Drumuri_m'] / aria_celula_m2
print("[LOG] 5. OSMnx complet.\n")

# ==========================================
# 6. LIVRABILE CP1, RAPORT DE ACOPERIRE & HARTĂ INTERACTIVĂ
# ==========================================
print("[LOG] 6. Generare rezultate finale...\n")

coloane_analiza = ['ID_Celula', 'LST_Mediu_C', 'NDVI_Mediu', 'Inaltime_GHSL_m', 'Fractie_Cladiri_OSM', 'Fractie_Parcuri_OSM', 'Densitate_Drumuri', 'geometry']

# --- ADAUGAT: RAPORT DE ACOPERIRE ---
print("=======================================")
print("=== LIVRABIL 2: RAPORT DE ACOPERIRE ===")
print("=======================================")
total_initial = len(grid_gdf)
coloane_raport = ['ID_Celula', 'LST_Mediu_C', 'NDVI_Mediu', 'Inaltime_GHSL_m', 'Fractie_Cladiri_OSM', 'Fractie_Parcuri_OSM', 'Densitate_Drumuri']

print("Valori lipsă (NaN) per sursă în gridul inițial:")
print(grid_gdf[coloane_raport].isna().sum())

matrice_X_gdf = grid_gdf.dropna(subset=['LST_Mediu_C', 'NDVI_Mediu', 'Inaltime_GHSL_m'])[coloane_analiza]
total_final = len(matrice_X_gdf)
supravietuire = (total_final / total_initial) * 100

print(f"\nRata de supraviețuire: {total_final} din {total_initial} celule de 500m au supraviețuit alinierii ({supravietuire:.1f}%).\n")
# ------------------------------------
# Afișare DataFrame text
df_afisare = matrice_X_gdf.drop(columns=['geometry'])
print("=======================================")
print("=== TABELUL ALINIAT (Eșantion 10 linii) ===")
print("=======================================")
print(df_afisare.sample(min(10, len(df_afisare)), random_state=42))

print("\n[LOG] Se calculează coordonatele pentru linkurile de Google Maps...")
# Extragem coordonatele centrului fiecărei celule, transformate în sistem GPS (EPSG:4326)
centru_gps = matrice_X_gdf.to_crs("EPSG:4326").geometry.centroid
matrice_X_gdf['Google_Maps'] = [
    f'<a href="https://www.google.com/maps?q={lat},{lon}" target="_blank">Deschide în Google Maps</a>'
    for lat, lon in zip(centru_gps.y, centru_gps.x)
]

print("[LOG] Se generează harta interactivă (poate dura câteva secunde)...")

# Construire hartă interactivă folosind fundal public Esri cu 20 de trepte
harta = matrice_X_gdf.explore(
    column="LST_Mediu_C",
    cmap="RdYlBu_r",
    scheme="Quantiles",
    k=20,                      # <--- 20 de trepte de culori
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    attr="Tiles &copy; Esri &mdash; Esri, DeLorme, NAVTEQ",
    tooltip=[
        "ID_Celula", "LST_Mediu_C", "NDVI_Mediu",
        "Inaltime_GHSL_m", "Fractie_Cladiri_OSM",
        "Fractie_Parcuri_OSM", "Densitate_Drumuri"
    ],
    popup=[
        "ID_Celula", "LST_Mediu_C", "Google_Maps"
    ],
    style_kwds={"weight": 1, "color": "black", "fillOpacity": 0.75},
    name="Grid București (500m)"
)

# Afișează harta direct în notebook
display(harta)
print("\n[LOG] Pipeline finalizat cu succes.")

[LOG] Inițializare și Autentificare Google Earth Engine...
[LOG] GEE Inițializat cu succes.

[LOG] 0. Se generează gridul de 500m pentru București...
[LOG] 0. Grid generat pe serverul GEE.

[LOG] 1. Interogare GEE pentru Landsat 9 LST...
[LOG] 1. Landsat 9 procesat.

[LOG] 2. Interogare GEE pentru Sentinel-2 NDVI...
[LOG] 2. Sentinel-2 procesat.

[LOG] 3. Interogare GEE pentru GHSL (Înălțime clădiri)...
[LOG] 3. GHSL procesat.

[LOG] 4. Se descarcă grid-ul satelitar din GEE în mediul local...
[LOG] 4. Descărcare completă. Tabel de bază: 856 rânduri (Proiecție curentă: EPSG:32635).

[LOG] 5. Începe interogarea vectorială prin OSMnx...
[LOG] 5.1. Descărcare clădiri de pe Overpass API...
[LOG] 5.1. Intersecție spațială clădiri vs. grid...
[LOG] 5.2. Descărcare parcuri de pe Overpass API...
[LOG] 5.2. Intersecție spațială parcuri vs. grid...
[LOG] 5.3. Descărcare drumuri de pe Overpass API...
[LOG] 5.3. Intersecție spațială drumuri vs. grid...
[LOG] 5.4. Integrare trăsături OSM în tabelul 


[LOG] Pipeline finalizat cu succes.
